# 01 — Inventarisasi Dataset dan EDA Awal KulitKu
**Tujuan:** manifest per file, kualitas teknis, duplikat, distribusi visual awal, dan antrean audit. Bukan cleaning final, split, diagnosis, atau training.

Dataset: Eczema (6.715), Herpes Zoster (8.082), Normal (7.758), Ringworm (8.129), total awal dilaporkan 30.684. Angka nyata dihitung dari folder. Gambar asli dan augmentasi tercampur, tidak ada ID pasien.

**Prinsip:** semua `original_status=unknown`, `original_id`/`group_id` kosong sampai ada bukti. `include` memakai tiga nilai: kosong=belum diputuskan, 0=tidak dipakai berdasarkan masalah teknis tercatat, 1=diterima setelah audit (tidak diberikan otomatis di notebook ini). Jangan mengganti kosong menjadi 1 atau 0 saat membaca CSV. Duplikat SHA bukan identitas pasien. Jangan menghapus foto sulit untuk menaikkan skor.

Notebook mandiri: tidak membutuhkan variabel notebook 00, repository atau GPU. Jalankan dari atas. Dataset sumber tidak dimodifikasi. Waktu scan puluhan ribu file bergantung I/O; checkpoint dibuat berkala.


## 1. Import dan hubungkan Drive
Gunakan paket yang sudah ada; tidak menimpa pandas/Pillow bawaan Colab. Jika import paket hilang, instal paket yang disebut lalu restart dan jalankan lagi dari atas. Tidak memerlukan timm, torch, imagehash, atau OpenCV untuk EDA ini.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, shutil, warnings, re, random, platform
import subprocess, importlib.metadata as metadata
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFile
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value)
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Di luar Colab: isi path konfigurasi sesuai filesystem lokal.')

# Jangan menerima file terpotong seolah-olah valid.
ImageFile.LOAD_TRUNCATED_IMAGES = False
print('Python:', platform.python_version())
print({p: metadata.version(p) for p in ['numpy','pandas','Pillow','matplotlib']})


## 2. Konfigurasi — periksa lokasi dataset
`DATA_SOURCE` langsung berisi keempat folder kelas. `PREFER_LOCAL_COPY=True` menggunakan salinan dari notebook 00 hanya bila daftar path/ukuran cocok dengan Drive. Kecocokan ukuran bukan bukti isi identik; scan tetap menghitung SHA256 setiap file. Jangan mengubah dataset selama scan.

Untuk melanjutkan scan terputus, isi `RESUME_RUN_ID` dengan ID folder run sebelumnya dan gunakan sumber serta konfigurasi yang sama. Saat resume, SHA file diperiksa ulang sebelum memakai hasil lama. Setiap run menyimpan laporan terpisah agar audit manual tidak tertimpa.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/KulitKu')
DATA_SOURCE = DRIVE_ROOT / 'datasets' / 'original'  # GANTI jika dataset berada di folder lain
LOCAL_DATA_CANDIDATE = Path('/content/kulitku-data/raw')
PREFER_LOCAL_COPY = True
RESUME_RUN_ID = ''  # contoh: 20261105T050000_123456Z; kosong = run baru
SEED = 42
CLASSES = ['Eczema','Herpes Zoster','Normal','Ringworm']
REPORTED_COUNTS = {'Eczema':6715,'Herpes Zoster':8082,'Normal':7758,'Ringworm':8129}
ALLOWED_EXTENSIONS = {'.jpg','.jpeg','.png','.webp'}
ALLOWED_FORMATS = {'JPEG','PNG','WEBP'}
MAX_PIXELS = 25_000_000
MIN_SIDE_PROVISIONAL = 128
CHECKPOINT_EVERY = 500
EXAMPLES_PER_CLASS = 8
SCANNER_VERSION = '1.0.0'
random.seed(SEED)
rng = np.random.default_rng(SEED)
# Pillow tetap mempertahankan perlindungan bomb; batas 25MP dicek sebelum decode.
Image.MAX_IMAGE_PIXELS = MAX_PIXELS

if not DATA_SOURCE.is_dir():
    hints = [Path('/content/drive/MyDrive/datasets/original'),
             Path('/content/drive/MyDrive/KulitKu/datasets/original')]
    print('Lokasi contoh yang ada:', [str(p) for p in hints if p.is_dir()])
    raise FileNotFoundError(f'DATA_SOURCE tidak ditemukan: {DATA_SOURCE}. Ubah konfigurasi dahulu.')
missing = [c for c in CLASSES if not (DATA_SOURCE/c).is_dir()]
if missing:
    print('Isi folder:', [p.name for p in DATA_SOURCE.iterdir()][:30])
    raise FileNotFoundError(f'Folder kelas tidak ditemukan: {missing}')

def path_size_map(base):
    return {p.relative_to(base).as_posix():p.stat().st_size
            for c in CLASSES for p in sorted((base/c).rglob('*')) if p.is_file() and not p.is_symlink()}

DATA_ROOT = DATA_SOURCE
if PREFER_LOCAL_COPY and all((LOCAL_DATA_CANDIDATE/c).is_dir() for c in CLASSES):
    if path_size_map(LOCAL_DATA_CANDIDATE) == path_size_map(DATA_SOURCE):
        DATA_ROOT = LOCAL_DATA_CANDIDATE
    else:
        print('Salinan lokal tidak cocok daftar/ukuran; memakai sumber Drive. Tidak menimpa cache.')
RUN_ID = RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
if not re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID):
    raise ValueError('RUN_ID hanya boleh huruf, angka, underscore, dan dash.')
RUN_DIR = DRIVE_ROOT / 'reports' / '01_inventory_eda' / RUN_ID
WORK_DIR = Path('/content/kulitku-inventory-work') / RUN_ID
FIG_DIR = WORK_DIR/'figures'
for folder in [RUN_DIR, WORK_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('Dataset yang dibaca:', DATA_ROOT)
print('Run ID:', RUN_ID)
print('Hasil permanen:', RUN_DIR)


## 3. Daftar file dan struktur folder
Inventarisasi seluruh file di bawah empat folder kelas, termasuk ekstensi tidak didukung. Folder/berkas tingkat atas di luar kelas dicatat terpisah, bukan diasumsikan bagian dataset. Folder kosong dicatat. Symlink tidak dibaca. `image_id` berasal dari SHA256 path relatif sehingga file byte-identik di dua path tetap memiliki ID berbeda.


In [ ]:
all_files = sorted([p for c in CLASSES for p in (DATA_ROOT/c).rglob('*') if p.is_file() or p.is_symlink()],
                   key=lambda p:p.relative_to(DATA_ROOT).as_posix())
if not all_files:
    raise RuntimeError('Tidak ada file ditemukan.')
file_list = pd.DataFrame([{'relative_path':p.relative_to(DATA_ROOT).as_posix(),
    'label':p.relative_to(DATA_ROOT).parts[0], 'extension':p.suffix.lower(),
    'is_symlink':p.is_symlink()} for p in all_files])
file_list.to_csv(WORK_DIR/'discovered_files.csv',index=False)
unexpected = [{'name':p.name,'kind':'directory' if p.is_dir() else 'file'}
              for p in sorted(DATA_ROOT.iterdir()) if p.name not in CLASSES]
pd.DataFrame(unexpected,columns=['name','kind']).to_csv(WORK_DIR/'unexpected_top_level.csv',index=False)
empty_dirs = [p.relative_to(DATA_ROOT).as_posix() for c in CLASSES
              for p in [DATA_ROOT/c,*sorted((DATA_ROOT/c).rglob('*'))]
              if p.is_dir() and not any(p.iterdir())]
pd.DataFrame({'empty_directory':empty_dirs}).to_csv(WORK_DIR/'empty_directories.csv',index=False)
display(file_list.groupby('label').size().reindex(CLASSES,fill_value=0).rename('semua_file').to_frame())
print('Total file:',len(all_files),'| Di luar kelas tingkat atas:',len(unexpected))


## 4. Fungsi inspeksi gambar
Setiap file di-hash SHA256, dibaca headernya, diverifikasi, dibuka ulang, dan didekode penuh bila memenuhi batas. Gambar lebih dari 25MP ditandai **tidak didekode**, bukan otomatis disebut korup. RGB untuk statistik menggunakan orientasi EXIF; alpha dikomposit ke putih. File sumber tidak disimpan ulang.

Statistik visual dihitung pada thumbnail maksimum 256px: brightness, kontras, saturasi, entropy, Laplacian, edge fraction, area hampir hitam/putih. Ini indikator eksplorasi, bukan penentu diagnosis atau kelayakan otomatis. Statistik multiframe hanya frame pertama dan perlu review.

`dhash_64` hanya petunjuk kemiripan. Hash sama dapat terjadi pada gambar sederhana berbeda. Tidak dipakai mengisi group_id atau membuang file.


In [ ]:
# Core functions embedded directly into the notebook; no repository import needed.
def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def read_display_rgb(path):
    # RGB conversion on a COPY; dataset source is never modified.
    with warnings.catch_warnings():
        warnings.simplefilter('error', Image.DecompressionBombWarning)
        with Image.open(path) as im:
            if im.width * im.height > MAX_PIXELS:
                raise ValueError('pixel limit exceeded')
            im.seek(0)
            im.load()
            image = ImageOps.exif_transpose(im)
            if image.mode in ('RGBA', 'LA') or 'transparency' in image.info:
                rgba = image.convert('RGBA')
                bg = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
                return Image.alpha_composite(bg, rgba).convert('RGB')
            return image.convert('RGB')


def inspect_file(path):
    rel = path.relative_to(DATA_ROOT).as_posix()
    label = Path(rel).parts[0]
    row = dict(image_id='img_' + hashlib.sha256(rel.encode()).hexdigest(),
        relative_path=rel, label=label, filename=path.name, extension=path.suffix.lower(),
        file_size_bytes=None, mtime_ns=None, width=None, height=None,
        display_width=None, display_height=None, format='', mode='', n_frames=None,
        megapixels=None, aspect_ratio=None, rgb_convertible=False,
        readability_status='not_checked', readable=None, error_type='', error_message='',
        sha256='', rgb_pixels_sha256='', dhash_64='', has_exif=False, has_alpha=False,
        filename_aug_hint=False, filename_aug_tokens='',
        original_status='unknown', original_status_evidence='',
        original_id='', group_id='', group_evidence='',
        include=None, review_status='pending', audit_note='',
        below_minimum=False, over_maximum=False, extension_format_mismatch=False,
        brightness_mean=None, brightness_std=None, saturation_mean=None,
        grayscale_entropy=None, laplacian_variance=None, edge_fraction=None,
        near_black_fraction=None, near_white_fraction=None,
        source_changed_during_scan=False)
    notes = []
    def finish():
        row['audit_note'] = '; '.join(notes)
        return row
    if path.is_symlink():
        row.update(readability_status='skipped_symlink', include=0)
        notes.append('Symlink tidak dibaca; verifikasi sumber dan salin file nyata bila layak.')
        return finish()
    try:
        before = path.stat()
        row['file_size_bytes'], row['mtime_ns'] = before.st_size, before.st_mtime_ns
        row['sha256'] = sha256_file(path)
    except OSError as exc:
        row.update(readability_status='io_error', include=0, error_type=type(exc).__name__,
                   error_message=str(exc)[:250])
        notes.append('File gagal diakses/dihash; tinjau lalu ulangi jika kesalahan I/O sementara.')
        return finish()
    tokens = sorted(set(re.findall(r'(?:^|[\W_])(aug(?:mented|mentation)?|rot(?:ate|ated)?|flip(?:ped)?|zoom|shear|brightness|contrast)(?=[\W_\d]|$)', path.stem.lower())))
    row['filename_aug_hint'], row['filename_aug_tokens'] = bool(tokens), ','.join(tokens)
    if tokens:
        notes.append('Token nama file hanya petunjuk augmentasi; bukan bukti status atau group_id.')
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        row.update(readability_status='unsupported_extension', include=0)
        notes.append('Ekstensi di luar protokol; tidak didekode. File tetap terinventarisasi.')
        return finish()
    try:
        with warnings.catch_warnings(record=True):
            warnings.simplefilter('always', Image.DecompressionBombWarning)
            with Image.open(path) as im:
                w, h = im.size
                row.update(width=w, height=h, format=im.format or '', mode=im.mode,
                           n_frames=getattr(im, 'n_frames', 1), megapixels=w*h/1e6,
                           aspect_ratio=w/h if h else None,
                           has_exif=bool(im.info.get('exif')),
                           has_alpha=im.mode in ('RGBA','LA') or 'transparency' in im.info)
                row['below_minimum'] = w < MIN_SIDE_PROVISIONAL or h < MIN_SIDE_PROVISIONAL
                row['over_maximum'] = w*h > MAX_PIXELS
                expected = {'.jpg':'JPEG','.jpeg':'JPEG','.png':'PNG','.webp':'WEBP'}[path.suffix.lower()]
                row['extension_format_mismatch'] = row['format'] != expected
                if row['over_maximum']:
                    row.update(readability_status='skipped_pixel_limit', include=0)
                    notes.append('Melebihi 25 MP; decode penuh tidak dicoba, bukan dinyatakan korup.')
                    return finish()
                if row['format'] not in ALLOWED_FORMATS:
                    row.update(readability_status='unsupported_content_format', include=0)
                    notes.append('Format isi di luar protokol walaupun ekstensi diizinkan.')
                    return finish()
                im.verify()
        # verify() alone is insufficient: reopen and force full pixel decode.
        rgb = read_display_rgb(path)
        row.update(readability_status='readable', readable=True, rgb_convertible=True,
                   display_width=rgb.width, display_height=rgb.height)
        pixels_h = hashlib.sha256()
        pixels_h.update(f'RGB:{rgb.width}x{rgb.height}:'.encode())
        pixels_h.update(rgb.tobytes())
        row['rgb_pixels_sha256'] = pixels_h.hexdigest()
        dh = np.asarray(rgb.convert('L').resize((9,8), Image.Resampling.LANCZOS), dtype=np.int16)
        bits = (dh[:,1:] > dh[:,:-1]).ravel()
        row['dhash_64'] = f'{sum(int(b) << i for i,b in enumerate(bits)):016x}'
        thumb = rgb.copy()
        thumb.thumbnail((256,256), Image.Resampling.LANCZOS)
        gray = np.asarray(thumb.convert('L'), dtype=np.float32)
        hsv = np.asarray(thumb.convert('HSV'), dtype=np.float32)
        hist = np.bincount(gray.astype(np.uint8).ravel(), minlength=256).astype(float)
        probs = hist[hist>0]/hist.sum()
        if min(gray.shape) >= 3:
            lap = (-4*gray[1:-1,1:-1]+gray[:-2,1:-1]+gray[2:,1:-1]+gray[1:-1,:-2]+gray[1:-1,2:])
            edge = float(np.mean(np.abs(lap)>20))
            lap_var = float(np.var(lap))
        else:
            lap_var, edge = None, None
        row.update(brightness_mean=float(gray.mean()), brightness_std=float(gray.std()),
            saturation_mean=float(hsv[:,:,1].mean()/255),
            grayscale_entropy=float(-(probs*np.log2(probs)).sum()),
            laplacian_variance=lap_var, edge_fraction=edge,
            near_black_fraction=float(np.mean(gray<=10)), near_white_fraction=float(np.mean(gray>=245)))
        if row['below_minimum']:
            notes.append('Di bawah batas sementara 128 per sisi; keputusan minimum belum dikunci, jangan otomatis buang.')
        if row['extension_format_mismatch']:
            notes.append('Ekstensi berbeda dari format isi; tinjau penamaan.')
        if row['n_frames'] > 1:
            notes.append('Multiframe: keterbacaan/statistik hanya frame pertama; perlu aturan anotasi sebelum dipakai.')
        if row['has_alpha']:
            notes.append('Ada alpha/transparansi; statistik RGB memakai komposit latar putih.')
        notes.append('Menunggu review label, relevansi kulit, original/augmentasi dan lineage; belum disetujui untuk split.')
        after = path.stat()
        if (before.st_size,before.st_mtime_ns) != (after.st_size,after.st_mtime_ns):
            row.update(source_changed_during_scan=True, include=0)
            notes.append('File berubah saat pemindaian; ulangi scan dengan dataset versi tetap.')
    except (Image.DecompressionBombError, Image.DecompressionBombWarning) as exc:
        row.update(readability_status='skipped_safety_limit', include=0,
                   error_type=type(exc).__name__, error_message=str(exc)[:250])
        notes.append('Pillow menolak karena batas keselamatan; tidak dianggap bukti file korup.')
    except Exception as exc:
        row.update(readability_status='decode_error', readable=False, include=0,
                   error_type=type(exc).__name__, error_message=str(exc)[:250])
        notes.append('Header/verify/full decode/konversi gagal; review dan ulangi bila perlu.')
    return finish()


def add_duplicate_flags(df):
    df = df.copy()
    for key, prefix in [('sha256','exact_duplicate'), ('rgb_pixels_sha256','pixel_duplicate')]:
        valid = df[key].fillna('').ne('')
        counts = df.loc[valid].groupby(key)[key].transform('size')
        labels = df.loc[valid].groupby(key)['label'].transform('nunique')
        df[prefix+'_count'] = 0
        df[prefix+'_label_conflict'] = False
        df[prefix+'_group_id'] = ''
        df.loc[valid, prefix+'_count'] = counts
        duplicated = valid & df[key].map(df.loc[valid,key].value_counts()).fillna(0).gt(1)
        df.loc[duplicated, prefix+'_group_id'] = prefix+'_'+df.loc[duplicated,key]
        df.loc[valid, prefix+'_label_conflict'] = labels.gt(1)
    for i, row in df.iterrows():
        extra = []
        if row['exact_duplicate_count'] > 1:
            extra.append('Byte-identik SHA256: perlu pilih representatif saat audit, belum otomatis dibuang.')
        elif row['pixel_duplicate_count'] > 1:
            extra.append('RGB frame pertama identik setelah orientasi/alpha; tinjau sebagai kandidat duplikat piksel.')
        if row['exact_duplicate_label_conflict'] or row['pixel_duplicate_label_conflict']:
            extra.append('Konflik label pada duplikat: karantina sampai review selesai.')
            df.at[i,'review_status'] = 'label_conflict'
        if extra:
            df.at[i,'audit_note'] = '; '.join([row['audit_note'], *extra]).strip('; ')
    # Critically, duplicate hashes do not populate lineage group_id or original_status.
    return df


## 5. Scan dengan checkpoint dan dukungan resume
Output checkpoint disalin ke Drive tiap 500 file. Jika runtime putus, sebagian kecil pekerjaan terakhir mungkin perlu diulang. Jangan menjalankan dua sesi ke run ID yang sama.

Resume memeriksa signature daftar file, versi scanner, konfigurasi dan versi Pillow/NumPy; jika tidak cocok, buat run baru. File yang berubah sejak scan lama diperiksa ulang. Error koneksi Drive menghentikan proses agar hasil tidak diklaim tersimpan.


In [ ]:
def atomic_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name+'.partial')
    shutil.copy2(src,tmp)
    if tmp.stat().st_size != src.stat().st_size:
        raise IOError('Ukuran salinan tidak sesuai.')
    os.replace(tmp,dst)

source_snapshot = []
for path in all_files:
    stat = path.lstat()
    source_snapshot.append([path.relative_to(DATA_ROOT).as_posix(),stat.st_size,stat.st_mtime_ns,path.is_symlink()])
signature_info = {'scanner_version':SCANNER_VERSION,'files':source_snapshot,
    'classes':CLASSES,'max_pixels':MAX_PIXELS,'minimum_provisional':MIN_SIDE_PROVISIONAL,
    'Pillow':metadata.version('Pillow'),'numpy':metadata.version('numpy'),
    'rgb_policy':'EXIF orientation; alpha composite white; frame0; thumbnail256'}
signature = hashlib.sha256(json.dumps(signature_info,sort_keys=True).encode()).hexdigest()
config_path = RUN_DIR/'scan-config.json'
if config_path.exists():
    old_config = json.loads(config_path.read_text())
    if old_config['signature'] != signature:
        raise RuntimeError('Sumber/konfigurasi berubah. Kosongkan RESUME_RUN_ID untuk run baru.')
else:
    config_path.write_text(json.dumps({'signature':signature,'run_id':RUN_ID,'seed':SEED,
        'data_source':str(DATA_SOURCE),'actual_read_root':str(DATA_ROOT),
        'reported_counts':REPORTED_COUNTS,'signature_info':signature_info},indent=2))
checkpoint = WORK_DIR/'scan_checkpoint.jsonl'
remote_checkpoint = RUN_DIR/'scan_checkpoint.jsonl'
if not checkpoint.exists() and remote_checkpoint.exists():
    shutil.copy2(remote_checkpoint,checkpoint)
cached = {}
if checkpoint.exists():
    lines = checkpoint.read_text().splitlines()
    for i,line in enumerate(lines):
        if not line.strip(): continue
        try:
            record = json.loads(line)
            cached[record['relative_path']] = record
        except (ValueError,KeyError):
            if i == len(lines)-1:
                print('Baris checkpoint terakhir tidak lengkap; akan dipindai ulang.')
            else:
                raise RuntimeError('Checkpoint rusak di tengah file; gunakan run baru.')
    # Rewrite complete records to remove truncated tail before appending.
    checkpoint.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in cached.values()))
records = []
with checkpoint.open('a',encoding='utf-8') as stream:
    for index,path in enumerate(tqdm(all_files,desc='Inventarisasi'),1):
        rel = path.relative_to(DATA_ROOT).as_posix()
        previous = cached.get(rel)
        use_cache = False
        if previous and not path.is_symlink() and previous.get('sha256'):
            try:
                use_cache = previous['sha256'] == sha256_file(path)
            except OSError:
                pass
        record = previous if use_cache else inspect_file(path)
        records.append(record)
        if not use_cache:
            stream.write(json.dumps(record,ensure_ascii=False)+'\n')
        if index % CHECKPOINT_EVERY == 0:
            stream.flush()
            atomic_copy(checkpoint,remote_checkpoint)
    stream.flush()
atomic_copy(checkpoint,remote_checkpoint)
df = add_duplicate_flags(pd.DataFrame(records))
df['include'] = pd.array(df['include'],dtype='Int64')
df['readable'] = pd.array(df['readable'],dtype='boolean')
for col in ['width','height','display_width','display_height','n_frames','file_size_bytes','mtime_ns']:
    df[col] = pd.array(df[col],dtype='Int64')
assert df.image_id.is_unique and df.relative_path.is_unique
assert df.original_status.eq('unknown').all() and df.group_id.eq('').all()
assert not df['include'].eq(1).fillna(False).any()
MANIFEST_NAME = 'manifest_inventory.csv'
df.to_csv(WORK_DIR/MANIFEST_NAME,index=False,encoding='utf-8')
atomic_copy(WORK_DIR/MANIFEST_NAME,RUN_DIR/MANIFEST_NAME)
display(df[['image_id','relative_path','label','width','height','format','readability_status','original_status','group_id','include']].head())
print('Manifest:',RUN_DIR/MANIFEST_NAME)
print('Kosong pada include = pending audit, bukan diterima.')


## 6. Ringkasan jumlah kelas dan perbandingan angka awal
Bedakan semua file, kandidat ekstensi gambar, dan gambar yang berhasil didekode. Selisih dengan angka awal tidak otomatis berarti data hilang: periksa file tambahan, subfolder, ekstensi, dan versi dataset. Persentase berbasis gambar belum mengukur keseimbangan pasien/kelompok independen.


In [ ]:
def save_table(frame,name):
    frame.to_csv(WORK_DIR/name,index=False)
    return frame

def save_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR/name,dpi=140,bbox_inches='tight')
    plt.show()
    plt.close()

valid = df.loc[df.readability_status.eq('readable')].copy()
class_summary = pd.DataFrame({'label':CLASSES})
class_summary['reported'] = class_summary.label.map(REPORTED_COUNTS)
class_summary['all_files'] = class_summary.label.map(df.label.value_counts()).fillna(0).astype(int)
class_summary['image_extension_files'] = class_summary.label.map(df.loc[df.extension.isin(ALLOWED_EXTENSIONS),'label'].value_counts()).fillna(0).astype(int)
class_summary['readable_images'] = class_summary.label.map(valid.label.value_counts()).fillna(0).astype(int)
class_summary['difference_from_reported'] = class_summary.image_extension_files-class_summary.reported
class_summary['readable_percent'] = class_summary.readable_images/max(len(valid),1)*100
save_table(class_summary,'class_summary.csv')
display(class_summary)
counts = class_summary.set_index('label')[['reported','image_extension_files','readable_images']]
counts.plot.bar(figsize=(10,4),rot=15)
plt.ylabel('Jumlah file/gambar');plt.title('Distribusi kelas: dilaporkan vs ditemukan vs terbaca')
save_figure('01_class_distribution.png')
positive_counts = class_summary.readable_images[class_summary.readable_images>0]
imbalance_ratio = float(positive_counts.max()/positive_counts.min()) if len(positive_counts)==len(CLASSES) else None
print('Rasio kelas terbesar/terkecil (gambar terbaca):',imbalance_ratio)


## 7. Keterbacaan, format, mode warna dan struktur direktori


In [ ]:
status_table = pd.crosstab(df.label,df.readability_status).reindex(CLASSES,fill_value=0)
save_table(status_table.reset_index(),'readability_by_class.csv')
display(status_table)
format_table = df.groupby(['label','extension','format','mode'],dropna=False).size().rename('count').reset_index()
save_table(format_table,'formats_modes.csv')
display(format_table.head(40))
mode_table = pd.crosstab(valid.label,valid['mode']).reindex(CLASSES,fill_value=0)
if len(valid):
    mode_table.plot.bar(stacked=True,figsize=(10,4),rot=15)
    plt.ylabel('Jumlah gambar');plt.title('Mode warna sebelum konversi RGB')
    save_figure('02_color_modes.png')
directory_table = df.assign(parent=df.relative_path.map(lambda x:Path(x).parent.as_posix())).groupby(['label','parent']).size().rename('count').reset_index()
save_table(directory_table,'directory_counts.csv')
problem_files = df.loc[df.readability_status.ne('readable') | df.extension_format_mismatch | df.over_maximum | df.source_changed_during_scan]
save_table(problem_files,'technical_problem_files.csv')
print('File dengan isu teknis:',len(problem_files))
print('EXIF terdeteksi:',int(df.has_exif.sum()),'| alpha:',int(df.has_alpha.sum()),'| multiframe:',int(df.n_frames.gt(1).fillna(False).sum()))
# Tidak mengekspor GPS/nama pemilik/EXIF lengkap.


## 8. Resolusi, rasio aspek, ukuran file dan batas teknis
Ukuran width/height dari header; display_width/display_height setelah orientasi EXIF. Minimum 128px masih sementara: hanya flag review, belum alasan eksklusi otomatis. Tidak menyimpulkan foto resolusi kecil pasti tidak berguna atau foto resolusi besar pasti berkualitas.


In [ ]:
numeric_columns = ['width','height','megapixels','aspect_ratio','file_size_bytes']
resolution_stats = valid.groupby('label')[numeric_columns].describe(percentiles=[.05,.25,.5,.75,.95])
resolution_stats.to_csv(WORK_DIR/'resolution_descriptive_statistics.csv')
display(resolution_stats)
limit_summary = df.groupby('label').agg(below_minimum=('below_minimum','sum'),over_maximum=('over_maximum','sum')).reindex(CLASSES)
save_table(limit_summary.reset_index(),'resolution_flags.csv')
display(limit_summary)
if len(valid):
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for label in CLASSES:
        v=valid[valid.label.eq(label)]
        axes[0,0].hist(v.megapixels.dropna().astype(float),bins=35,alpha=.45,label=label)
        axes[0,1].hist(v.aspect_ratio.dropna().astype(float),bins=35,alpha=.45,label=label)
    axes[0,0].set(xlabel='Megapiksel',ylabel='Jumlah',title='Resolusi');axes[0,0].legend(fontsize=8)
    axes[0,1].set(xlabel='Lebar/tinggi',ylabel='Jumlah',title='Rasio aspek')
    sample=valid.sample(n=min(len(valid),5000),random_state=SEED)
    for label in CLASSES:
        s=sample[sample.label.eq(label)]
        axes[1,0].scatter(s.width.astype(float),s.height.astype(float),s=6,alpha=.25,label=label)
    axes[1,0].set(xscale='log',yscale='log',xlabel='Lebar (log)',ylabel='Tinggi (log)',title='Resolusi; sampel maksimal 5.000')
    axes[1,0].axvline(MIN_SIDE_PROVISIONAL,color='black',linestyle='--');axes[1,0].axhline(MIN_SIDE_PROVISIONAL,color='black',linestyle='--')
    axes[1,1].hist(valid.file_size_bytes.astype(float)/1024,bins=50)
    axes[1,1].set(xlabel='Ukuran file KiB',ylabel='Jumlah',title='Ukuran file')
    save_figure('03_resolution_aspect_size.png')
save_table(df.loc[df.below_minimum | df.over_maximum],'resolution_review_queue.csv')


## 9. Duplikat byte, duplikat piksel dan konflik label
- SHA256 sama: file byte-identik; laporkan grup, file terlibat dan redundansi n−1.
- RGB pixel hash sama: tampilan frame pertama identik setelah EXIF/RGB/alpha, meskipun encoding berbeda; perlu review.
- Hash tersebut disimpan dalam kolom duplikat terpisah, bukan `group_id` lineage.
- Label berbeda pada duplikat wajib diselesaikan sebelum split.

Notebook tidak memilih representatif otomatis karena status asli belum diketahui. Jangan menjumlahkan redundansi byte dan piksel: kedua himpunan dapat tumpang tindih.


In [ ]:
duplicate_summary = []
for key,prefix in [('sha256','exact_duplicate'),('rgb_pixels_sha256','pixel_duplicate')]:
    members = df[df[prefix+'_count'].gt(1)].copy()
    groups = members.groupby(key).agg(n_files=('image_id','size'),n_labels=('label','nunique'),
        labels=('label',lambda x:' | '.join(sorted(set(x)))),
        paths=('relative_path',lambda x:' | '.join(x))).reset_index()
    save_table(members,prefix+'_members.csv')
    save_table(groups,prefix+'_groups.csv')
    save_table(members[members[prefix+'_label_conflict']],prefix+'_label_conflicts.csv')
    duplicate_summary.append({'type':prefix,'groups':len(groups),'involved_files':len(members),
        'redundant_files':int((groups.n_files-1).sum()) if len(groups) else 0,
        'cross_label_groups':int(groups.n_labels.gt(1).sum()) if len(groups) else 0})
duplicate_summary=pd.DataFrame(duplicate_summary)
save_table(duplicate_summary,'duplicate_summary.csv')
display(duplicate_summary)
# Byte-unique count is descriptive, not final cleaned count.
byte_unique_by_class = df[df.sha256.ne('')].groupby('label').sha256.nunique().reindex(CLASSES,fill_value=0)
save_table(byte_unique_by_class.rename('unique_sha256_within_class').reset_index(),'byte_unique_by_class.csv')


## 10. Pola nama file dan petunjuk augmentasi
Tampilkan token dan prefix nama file untuk membantu penelusuran sumber. Kata aug/flip/rotate dapat salah positif; tidak adanya kata tersebut tidak berarti original. Tidak ada status atau grup yang diisi berdasarkan regex.

Bucket dHash identik disediakan sebagai kandidat review **tanpa pencarian semua pasangan**; ini bukan deteksi semua near-duplicate dan dapat memiliki false positive/negative. Foto polos berbeda dapat memiliki hash sama.


In [ ]:
aug_summary = df.groupby('label').agg(files=('image_id','size'),filename_hint=('filename_aug_hint','sum')).reindex(CLASSES)
save_table(aug_summary.reset_index(),'filename_hint_summary.csv')
display(aug_summary)
save_table(df.loc[df.filename_aug_hint,['image_id','relative_path','label','filename_aug_tokens','original_status','group_id']],'filename_augmentation_hints.csv')
prefixes = df.assign(filename_prefix=df.filename.map(lambda x:re.split(r'[_\-\s]',Path(x).stem)[0][:40]))
prefix_counts = prefixes.groupby(['label','filename_prefix']).size().rename('count').reset_index().sort_values('count',ascending=False)
save_table(prefix_counts,'filename_prefix_counts.csv')
display(prefix_counts.head(30))
dhash_counts = valid.loc[valid.dhash_64.ne('')].groupby('dhash_64').agg(n_files=('image_id','size'),n_pixel_hashes=('rgb_pixels_sha256','nunique'))
candidate_hashes=dhash_counts.index[(dhash_counts.n_files>1)&(dhash_counts.n_pixel_hashes>1)]
save_table(valid[valid.dhash_64.isin(candidate_hashes)],'same_dhash_review_candidates.csv')
print('Bucket dHash identik dengan pixel hash berbeda:',len(candidate_hashes))
print('Original terverifikasi:',int(df.original_status.eq('verified_original').sum()),'| Group lineage terisi:',int(df.group_id.ne('').sum()))


## 11. Indikator kualitas visual dan perbedaan antar kelas
Brightness memakai skala 0–255; saturasi 0–1; entropy grayscale dan Laplacian berasal dari thumbnail. Nilai Laplacian tidak dapat dipakai sebagai ambang blur universal. Perbedaan latar, tekstur, resolusi, watermark, atau kompresi dapat memengaruhinya.

Pilih contoh ekstrem untuk ditinjau, bukan otomatis dibuang. Jangan menaksir variasi warna kulit pasien dari brightness foto; pencahayaan dan latar memengaruhi pengukuran.


In [ ]:
quality_columns=['brightness_mean','brightness_std','saturation_mean','grayscale_entropy',
    'laplacian_variance','edge_fraction','near_black_fraction','near_white_fraction']
quality_stats=valid.groupby('label')[quality_columns].describe(percentiles=[.05,.5,.95])
quality_stats.to_csv(WORK_DIR/'visual_quality_descriptive_statistics.csv')
if len(valid):
    fig,axes=plt.subplots(2,3,figsize=(15,8))
    for ax,col in zip(axes.ravel(),quality_columns[:6]):
        arrays=[valid.loc[valid.label.eq(c),col].dropna().astype(float).values for c in CLASSES]
        ax.boxplot(arrays,labels=CLASSES,showfliers=False)
        ax.tick_params(axis='x',rotation=25,labelsize=8)
        ax.set_title(col,fontsize=10)
        if col=='laplacian_variance':ax.set_yscale('symlog',linthresh=1)
    save_figure('04_visual_quality_by_class.png')
quality_candidates=[]
for label in CLASSES:
    subset=valid[valid.label.eq(label)]
    for metric,ascending,reason in [('brightness_mean',True,'brightness_low'),('brightness_mean',False,'brightness_high'),
                                   ('laplacian_variance',True,'low_laplacian_review'),('grayscale_entropy',True,'low_entropy_review')]:
        selected=subset.dropna(subset=[metric]).sort_values([metric,'relative_path'],ascending=[ascending,True]).head(5).copy()
        selected['candidate_reason']=reason
        quality_candidates.append(selected)
quality_queue=pd.concat(quality_candidates,ignore_index=True) if quality_candidates else valid.assign(candidate_reason='')
save_table(quality_queue,'visual_extremes_review_queue.csv')


## 12. Galeri acak setiap kelas dan contoh untuk review
Galeri maksimum delapan gambar per kelas dipilih dengan seed tetap; tidak mengubah file. Foto EDA bukan bukti representatif seluruh populasi. Preview disimpan sebagai PNG tanpa metadata sumber; jangan unggah contact sheet ke GitHub publik jika hak penggunaan atau privasi belum jelas.


In [ ]:
def contact_sheet(rows,title,filename,columns=4):
    rows=rows.reset_index(drop=True)
    if rows.empty:
        print('Tidak ada contoh:',title)
        return
    n=len(rows); nrows=(n+columns-1)//columns
    fig,axes=plt.subplots(nrows,columns,figsize=(columns*3.2,nrows*3.3),squeeze=False)
    for ax in axes.ravel():ax.axis('off')
    for ax,(_,row) in zip(axes.ravel(),rows.iterrows()):
        try:
            picture=read_display_rgb(DATA_ROOT/row.relative_path)
            picture.thumbnail((512,512),Image.Resampling.LANCZOS)
            ax.imshow(picture)
            ax.set_title(f'{row.label} | {row.width}x{row.height}\n{row.image_id[:16]}',fontsize=8)
        except Exception as exc:
            ax.text(.1,.5,type(exc).__name__,fontsize=8)
    fig.suptitle(title,fontsize=12)
    save_figure(filename)

example_rows=[]
for i,label in enumerate(CLASSES):
    candidates=valid[valid.label.eq(label)]
    sample=candidates.sample(n=min(EXAMPLES_PER_CLASS,len(candidates)),random_state=SEED)
    example_rows.append(sample.assign(example_kind='random_by_class'))
    contact_sheet(sample,f'Contoh acak: {label}',f'05_random_class_{i}.png')
small=valid[valid.below_minimum].sort_values(['width','height','relative_path']).head(8)
contact_sheet(small,'Resolusi di bawah batas sementara: review, bukan otomatis dibuang','06_small_images.png')
low_lap=valid.dropna(subset=['laplacian_variance']).sort_values(['laplacian_variance','relative_path']).head(8)
contact_sheet(low_lap,'Laplacian rendah: tinjau ketajaman/latar/tekstur','07_low_laplacian.png')
conflicts=valid[valid.exact_duplicate_label_conflict | valid.pixel_duplicate_label_conflict].head(8)
contact_sheet(conflicts,'Konflik label pada duplikat: perlu review','08_label_conflicts.png')
# Sample members from at most three exact duplicate groups.
first_groups=df.loc[df.exact_duplicate_count.gt(1),'exact_duplicate_group_id'].drop_duplicates().head(3)
duplicate_examples=valid[valid.exact_duplicate_group_id.isin(first_groups)].groupby('exact_duplicate_group_id',sort=True).head(3).head(9)
contact_sheet(duplicate_examples,'Contoh file byte-identik; tidak menyatakan gambar asli','09_duplicate_examples.png',columns=3)
if example_rows:
    save_table(pd.concat(example_rows,ignore_index=True),'random_gallery_index.csv')

def preview_image(image_id):
    selected=df[df.image_id.eq(image_id)]
    if selected.empty:raise KeyError('image_id tidak ditemukan.')
    display(selected[['image_id','relative_path','label','audit_note']])
    image=read_display_rgb(DATA_ROOT/selected.iloc[0].relative_path)
    image.thumbnail((800,800),Image.Resampling.LANCZOS)
    display(image)
# Untuk review: preview_image('img_...ID lengkap dari CSV...')


## 13. Buat antrean review dan template audit manual
Manifest otomatis dipertahankan. Edit salinan `manual_audit_template.csv`, bukan file original. Isi reviewer/date dan bukti untuk setiap perubahan. Notebook ini tidak menerapkan template secara otomatis.

Aturan untuk notebook audit berikutnya:
- `original_status`: unknown / verified_original / verified_augmented; verifikasi memerlukan original_status_evidence.
- `group_id` hanya diisi dengan group_evidence; jangan memberi ID pasien fiktif atau group berbeda untuk tiap file agar CV bisa jalan.
- `include=1` hanya setelah review label/relevansi/kelayakan; persetujuan teknis tidak sama dengan siap split. Lineage yang belum selesai tetap harus ditahan dari eksperimen utama.
- Konflik label duplikat harus diselesaikan; pilih satu representatif dengan original diprioritaskan jika diketahui.
- Setiap perubahan label, include, original_status atau group harus memiliki catatan dan jejak audit. Jangan menerapkan label baru hanya karena prediksi model.


In [ ]:
manual_cols=['image_id','relative_path','label','original_status','original_status_evidence','original_id','group_id',
             'group_evidence','include','review_status','audit_note']
manual=df[manual_cols].copy()
manual['reviewer']='';manual['reviewed_at_utc']='';manual['label_evidence']=''
manual['proposed_label']='';manual['visual_skin_relevant']='';manual['watermark_or_text']=''
manual.to_csv(WORK_DIR/'manual_audit_template.csv',index=False)
review_mask=(df.readability_status.ne('readable')|df.below_minimum|df.extension_format_mismatch|
    df.exact_duplicate_count.gt(1)|df.pixel_duplicate_count.gt(1)|df.filename_aug_hint|
    df.n_frames.gt(1).fillna(False)|df.has_alpha)
queue=df.loc[review_mask].copy()
queue['priority']=np.where(queue.exact_duplicate_label_conflict|queue.pixel_duplicate_label_conflict,1,
    np.where(queue.readability_status.ne('readable'),2,3))
save_table(queue.sort_values(['priority','label','relative_path']),'priority_review_queue.csv')
print('Antrean prioritas:',len(queue),'file; SEMUA file tetap perlu audit, termasuk yang tidak terflag.')
# CSV round trip preserving pending include; no .fillna(1).
check_df=pd.read_csv(WORK_DIR/MANIFEST_NAME,dtype={'image_id':'string','sha256':'string','group_id':'string','include':'Int64'})
assert len(check_df)==len(df)
assert check_df['include'].isna().sum()==df['include'].isna().sum()
assert not check_df['include'].eq(1).fillna(False).any()


## 14. Data dictionary, laporan akhir, pemeriksaan dan backup
Backup mencakup CSV, grafik, laporan, konfigurasi, versi lingkungan dan checksum berkas hasil. Manifest final ini adalah **inventory**, bukan manifest siap training. Notebook tidak membuat split atau mengklaim data bersih.


In [ ]:
dictionary={
 'image_id':'img_ + SHA256 path relatif; identitas baris/file, bukan identitas pasien',
 'relative_path':'Path relatif terhadap akar dataset yang berisi empat kelas; gunakan ini saat pindah mesin',
 'label':'Label folder sumber; belum diverifikasi secara klinis',
 'width':'Lebar header sebelum EXIF; piksel', 'height':'Tinggi header sebelum EXIF; piksel',
 'format':'Format isi hasil Pillow, bukan sekadar ekstensi',
 'readability_status':'readable/decode_error/io_error/unsupported_extension/unsupported_content_format/skipped_pixel_limit/skipped_safety_limit/skipped_symlink',
 'readable':'true=decode dan RGB berhasil; false=decode gagal; kosong=tidak diperiksa/dibatasi',
 'sha256':'SHA256 seluruh byte file; kosong bila gagal membaca; bukan hash thumbnail',
 'rgb_pixels_sha256':'Hash dimensi+pixel RGB frame0 setelah EXIF/alpha; bukan hash byte',
 'original_status':'unknown pada inventarisasi; status verified memerlukan bukti audit',
 'original_status_evidence':'Dasar verifikasi asli/augmentasi; kosong jika belum diketahui',
 'original_id':'Identitas gambar asal bila terverifikasi; bukan tebakan dari filename',
 'group_id':'Kelompok lineage/pasien terverifikasi; kosong pada inventarisasi',
 'group_evidence':'Bukti hubungan kelompok; patient ID tidak tersedia pada dataset saat ini',
 'include':'Nullable integer: kosong=pending, 0=tidak dipakai dengan alasan, 1=disetujui setelah audit; bukan flag siap-split',
 'audit_note':'Alasan teknis/flag/keputusan; catatan otomatis bukan review klinis',
 'review_status':'pending atau label_conflict pada inventarisasi; bukan audit approved',
 'filename_aug_hint':'Petunjuk regex untuk review saja, tidak mengubah original_status/group_id',
 'dhash_64':'dHash grayscale 9x8; kandidat kemiripan, bukan bukti asal/pasien',
 'exact_duplicate_group_id':'Grup byte identik berdasarkan SHA; terpisah dari group_id lineage',
 'pixel_duplicate_group_id':'Grup RGB identik frame pertama; alpha putih dan EXIF diterapkan',
 'below_minimum':'Lebar/tinggi <128 sementara; review saja',
 'over_maximum':'Piksel >25MP; decode sengaja tidak dilakukan',
 'brightness_mean':'Rata-rata grayscale thumbnail, 0-255; bukan warna kulit pasien',
 'brightness_std':'Standar deviasi grayscale thumbnail; kontras deskriptif',
 'saturation_mean':'Saturasi rata-rata HSV thumbnail, 0-1',
 'grayscale_entropy':'Entropy grayscale thumbnail dalam bit',
 'laplacian_variance':'Varians operator Laplacian 4-neighbor pada thumbnail maksimum 256px; bukan ambang blur universal',
 'edge_fraction':'Fraksi pixel interior dengan |Laplacian|>20 pada thumbnail',
 'near_black_fraction':'Fraksi gray<=10 pada thumbnail; bukan detektor watermark/artefak',
 'near_white_fraction':'Fraksi gray>=245 pada thumbnail; bukan detektor watermark/artefak',
 'source_changed_during_scan':'Perubahan size/mtime saat inspeksi; harus scan ulang',
 'has_exif':'Indikasi metadata EXIF; isi sensitif tidak diekspor',
 'n_frames':'Jumlah frame; hanya frame0 didekode untuk statistik, multiframe perlu audit'
}
# Every manifest column gets a dictionary row, including supplementary metadata.
for name in df.columns:
    dictionary.setdefault(name,'Metadata pendukung; lihat implementasi fungsi inspect_file/add_duplicate_flags di notebook.')
pd.DataFrame([{'column':k,'definition':v} for k,v in dictionary.items()]).to_csv(WORK_DIR/'data_dictionary.csv',index=False)
environment={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,
    'platform':platform.platform(),'scanner_version':SCANNER_VERSION,'seed':SEED,
    'packages':{p:metadata.version(p) for p in ['numpy','pandas','Pillow','matplotlib']},
    'gpu_required':False,'dataset_signature':signature,'audit_approved':False,'split_created':False}
(WORK_DIR/'environment.json').write_text(json.dumps(environment,indent=2))
summary={'total_files':len(df),'readable_images':len(valid),
    'include_pending':int(df['include'].isna().sum()),'include_0_technical':int(df['include'].eq(0).fillna(False).sum()),
    'original_status_unknown':int(df.original_status.eq('unknown').sum()),
    'lineage_groups_verified':0,'imbalance_ratio_images':imbalance_ratio,
    'below_minimum_flagged':int(df.below_minimum.sum()),
    'filename_hints':int(df.filename_aug_hint.sum()),'unexpected_top_level_items':len(unexpected),
    'duplicates':duplicate_summary.to_dict('records'),'audit_ready_for_split':False}
(WORK_DIR/'summary.json').write_text(json.dumps(summary,indent=2))
class_lines='\n'.join(f'| {r.label} | {r.reported} | {r.all_files} | {r.readable_images} |' for r in class_summary.itertuples())
report=f"""# Laporan EDA Awal KulitKu

Run: {RUN_ID}. Seed: {SEED}. Scanner: {SCANNER_VERSION}.

## Jumlah data
| Kelas | Dilaporkan | Semua file | Terbaca |
|---|---:|---:|---:|
{class_lines}

Total file {len(df)}; gambar terbaca {len(valid)}; rasio kelas terbesar/terkecil berdasarkan gambar terbaca {imbalance_ratio}.

## Audit awal
- include pending: {summary['include_pending']}; include=0 karena isu teknis: {summary['include_0_technical']}.
- Semua {len(df)} status original/augmentasi masih unknown; group_id lineage belum diisi.
- Flag ukuran di bawah minimum sementara: {summary['below_minimum_flagged']}; bukan eksklusi otomatis.
- Petunjuk nama augmentasi: {summary['filename_hints']}; bukan bukti augmentasi.
- Lihat duplicate_summary.csv dan tabel konflik label. Duplikat byte dan piksel dapat tumpang tindih.

## Cara membaca grafik
Grafik brightness/entropy/Laplacian menggambarkan thumbnail, bukan diagnosis, tingkat penyakit, atau kelompok warna kulit. Galeri acak dan ekstrem perlu diperiksa manual untuk relevansi kulit, anotasi, watermark, latar, dan kemungkinan shortcut. Tidak ada deteksi otomatis watermark atau validasi klinis label.

## Batas kesimpulan
Data bersumber dari folder yang dilaporkan pengguna; versi/lisensi/sumber anotasi perlu didokumentasikan. Tidak ada ID pasien, lineage augmentasi belum terverifikasi, dan tidak ada external test. Keseimbangan jumlah gambar tidak membuktikan keseimbangan sampel independen. Statistik readable mengecualikan file yang tidak didekode karena safety limit, sehingga bukan deskripsi seluruh file.

## Tindakan berikutnya
1. Review file teknis gagal dan kemungkinan masalah I/O sementara.
2. Selesaikan konflik label pada duplikat dan pilih representatif dengan original diprioritaskan bila diketahui.
3. Telusuri bukti original/augmentasi dan kelompok, jangan membuat patient ID fiktif.
4. Tinjau relevansi, watermark/teks, dan contoh kualitas ekstrem tanpa menolak foto sulit semata.
5. Kunci kebijakan minimum sebelum split dan catat dampak eksklusi.
6. Buat manifest audit versi baru. Jangan melakukan split atau training dari manifest inventory ini.

## Berkas penting
manifest_inventory.csv, manual_audit_template.csv, priority_review_queue.csv, technical_problem_files.csv, data_dictionary.csv, summary.json, environment.json, figures/.
"""
(WORK_DIR/'EDA_REPORT.md').write_text(report)
assert set(CLASSES)==set(df.label.unique()) or set(df.label.unique()).issubset(set(CLASSES))
assert not df['include'].eq(1).fillna(False).any()
assert df.original_status.eq('unknown').all() and df.group_id.eq('').all()
# Results-only checksums; dataset file hashes already reside in manifest.
checksums={p.relative_to(WORK_DIR).as_posix():sha256_file(p) for p in sorted(WORK_DIR.rglob('*'))
           if p.is_file() and p.name!='output_checksums.json'}
(WORK_DIR/'output_checksums.json').write_text(json.dumps(checksums,indent=2))
for p in sorted(WORK_DIR.rglob('*')):
    if p.is_file():atomic_copy(p,RUN_DIR/p.relative_to(WORK_DIR))
# A stable pointer to a versioned file, not an overwrite of prior audit manifests.
manifest_pointer_dir=DRIVE_ROOT/'manifests'
manifest_pointer_dir.mkdir(parents=True,exist_ok=True)
(manifest_pointer_dir/'latest_inventory.json').write_text(json.dumps({
    'run_id':RUN_ID,'manifest_path':str(RUN_DIR/MANIFEST_NAME),
    'sha256':sha256_file(WORK_DIR/MANIFEST_NAME),'stage':'inventory_only','audit_approved':False},indent=2))
print('SELESAI. Hasil:',RUN_DIR)
print('Manifest:',RUN_DIR/MANIFEST_NAME)
print('Laporan:',RUN_DIR/'EDA_REPORT.md')
print('Belum siap split/training: original_status dan group_id masih menunggu audit.')
display(pd.DataFrame([summary]).drop(columns=['duplicates']))


## 15. Membuka manifest pada notebook berikutnya
Gunakan dtype nullable agar include kosong tetap pending. Isi group_id kosong tidak boleh diganti image_id untuk memaksakan grouped split. Baca laporan dan lanjutkan ke audit/cleaning; keputusan final disimpan pada manifest baru.

```python
import json
import pandas as pd
from pathlib import Path
pointer = json.loads((DRIVE_ROOT/'manifests/latest_inventory.json').read_text())
inventory = pd.read_csv(pointer['manifest_path'], dtype={
    'image_id':'string', 'sha256':'string', 'group_id':'string', 'include':'Int64'
})
```

Simpan notebook di GitHub: `machine-learning/notebooks/01_inventory_eda.ipynb`. Hasil EDA dan dataset tetap di Drive. Contact sheet bisa memuat foto sensitif: jangan publikasikan tanpa dasar izin.

**Referensi implementasi:** [Pillow Image](https://pillow.readthedocs.io/en/stable/reference/Image.html), [Pillow ImageOps](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html), [pandas nullable integer](https://pandas.pydata.org/docs/user_guide/integer_na.html). Image.open membaca secara lazy, sehingga pemeriksaan keterbacaan memakai verify lalu reopen/load. Aturan 25MP/128px adalah kebijakan penelitian, bukan ambang klinis dari dokumentasi.
